In [0]:
%run "../0_common/01. env-config"

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
        .withColumn("session_type", F.lit("RACE"))
        .drop("ingestion_timestamp","source_file","race_name","race_date")
)

sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
        .withColumn("session_type", F.lit("SPRINT"))
        .drop("ingestion_timestamp","source_file","race_name","race_date")
)

In [0]:
result_sprint_df = results_df.unionByName(sprints_df)
display(result_sprint_df.filter("season = 2025"))

In [0]:
fact_session_results_df = (
    result_sprint_df
        .withColumn("is_win", F.col("finish_position") == 1)
        .withColumn("is_podium", F.col("finish_position").between(1,3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_results_df.head(5))

In [0]:
fact_session_results_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
display(spark.table(target_table).limit(5))